# 03 · Recuperación y filtros

**§3.3 del enunciado**: una interfaz común que recibe una consulta y
devuelve resultados normalizados (`product_id`, posición, título,
metadatos y score con semántica declarada), búsqueda global con top-k
configurable, filtro de marca **ejecutado por la base de datos** y
tratamiento explícito de los tres casos límite: colección vacía, filtro
sin resultados y proveedor no disponible.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
print(f"Root del proyecto: {project_root}")


Root del proyecto: /home/manu/dev/github.com/manupm87/pontia-bd-vect


In [2]:
import json
import os

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TQDM_DISABLE", "1")

import pandas as pd
import plotly.io as pio
from dotenv import load_dotenv

from aurum_discovery import load_run_config

load_dotenv(project_root / ".env")
pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 90)
run_config = load_run_config()
print(f"Configuración final: {run_config.embedding_configuration}")


Configuración final: e5_small_title


In [3]:
from aurum_discovery import CatalogVectorStore, load_embedding_set

embedding_set = load_embedding_set(run_config.embedding_configuration)
store = CatalogVectorStore(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
    api_key=os.getenv("QDRANT_API_KEY", ""),
    collection_name=os.getenv("QDRANT_COLLECTION", "aurum-market-eval-catalogo"),
    vector_size=embedding_set.configuration.dimension,
    hnsw=run_config.hnsw,
    ef_search=run_config.ef_search,
)
store.ping()
print(f"Colección: {store.collection_name} · registros: {store.count()}")


Colección: aurum-market-eval-catalogo · registros: 15000


## La interfaz común

La interfaz común tiene dos niveles, y conviene distinguirlos. El
**contrato de resultados** (`SearchHit` normalizado + `store.search`, con
el filtro como parámetro) lo usan *todos* los consumidores: la CLI, los
scripts de evaluación y estos notebooks. Sobre él, `DiscoveryService` es
la **fachada de texto libre**: aplica el prefijo `query:` del contrato
E5, codifica y delega en la base — es lo que usa la CLI
(`make search q="..."`). Los scripts de evaluación, en cambio, entran
por `store.search` con los **embeddings precomputados** de
`data/embeddings/` — deliberadamente: así los artefactos entregados no
dependen de recodificar las consultas en cada ejecución y son
reproducibles bit a bit. Ambos caminos convergen en el mismo contrato.


In [4]:
from aurum_discovery import DiscoveryService, get_configuration

service = DiscoveryService(
    store=store,
    configuration=get_configuration(run_config.embedding_configuration),
)
hits = service.search_text("taladro sin cable potente", top_k=5)
pd.DataFrame([hit.as_dict() for hit in hits])[
    ["rank", "product_id", "title", "brand", "native_score", "score_kind", "higher_is_better"]
].assign(title=lambda frame: frame["title"].str.slice(0, 60))


,rank,product_id,title,brand,native_score,score_kind,higher_is_better
0,1,B07C2TM76Y,TEENO taladro Percutor sin cable 21V+2 baterías de iones de,TEENO,0.891467,similarity,True
1,2,B07JHCZ1T4,"Flybiz 1650/min Taladro Atornillador 21V, Broca，Taladro sin",FLY BIZ,0.885451,similarity,True
2,3,B01N5T6SL4,BLACK+DECKER BL188KB-QW - Taladro Percutor Motor Brushless c,Black+Decker,0.884235,similarity,True
3,4,B01A5VQHBY,Taladro Percutor Brushless 20V Worx WX373,WORX,0.883183,similarity,True
4,5,B0071T3MOO,"Bosch 12V System GSB 12V-15 - Taladro Percutor a Batería, 30",Bosch Professional,0.880980,similarity,True


In [5]:
branded_hits = service.search_text(
    "zapatillas cómodas para salir a correr", top_k=5, brand="NIKE"
)
assert {hit.brand for hit in branded_hits} == {"NIKE"}
pd.DataFrame(
    [
        {"rank": hit.rank, "score": round(hit.native_score, 4),
         "brand": hit.brand, "title": hit.title[:60]}
        for hit in branded_hits
    ]
)


,rank,score,brand,title
0,1,0.8821,NIKE,Nike ACG React Terra Gobe Trail Zapatillas de correr para ho
1,2,0.8808,NIKE,"Nike Wmns Air Zoom Pegasus 34, Zapatillas de Running Mujer,"
2,3,0.8803,NIKE,"Nike MD Runner 2 (GS), Zapatillas de Correr Mujer, Negro (Bl"
3,4,0.8781,NIKE,"Nike MD Runner 2 (GS), Zapatillas de Correr Unisex Adulto, N"
4,5,0.8780,NIKE,"Nike MD Runner 2 GS, Zapatillas de Correr Mujer, Negro (Blac"


El resultado normalizado **conserva el score nativo y su semántica**.
Este detalle importa más de lo que parece: cada motor vectorial devuelve
su score en una moneda distinta — Qdrant con métrica coseno devuelve una
*similitud* (mayor es mejor), pero Chroma o Weaviate devuelven una
*distancia* (menor es mejor), a veces bajo la misma etiqueta «cosine».
Convertir una en otra «a ojo», o comparar scores de motores distintos
como si fueran equivalentes, produce rankings sin sentido. Por eso el
contrato lleva el score acompañado de su declaración
(`score_kind="similarity"`, `higher_is_better=true`) en lugar de un
número desnudo.

## Filtro de marca dentro de la consulta: prefiltrar, no tachar

Hay dos maneras de combinar una búsqueda vectorial con una condición de
metadatos, y no son equivalentes:

- **Post-filtrado**: recuperar el top-k global y tachar lo que no cumple.
  Barato, pero roto por diseño — si solo 3 de los 10 mejores globales son
  de la marca, se devuelven 3 resultados (o ninguno), aunque el catálogo
  tenga cientos de productos de esa marca perfectamente válidos.
- **Prefiltrado (filtro en la consulta)**: la condición entra en el
  motor, que restringe el universo de búsqueda a los puntos que cumplen
  el filtro y devuelve el top-k *de ese universo*. Siempre hay k
  resultados si existen k candidatos.

El enunciado exige lo segundo, y el índice de payload *keyword* sobre
`brand` (notebook 02) es lo que lo hace eficiente: sin él, el motor
tendría que escanear payloads uno a uno. En Qdrant la condición viaja
como `query_filter` de `query_points`, y el grafo HNSW la tiene en
cuenta durante la navegación. Las cuatro consultas oficiales,
verificadas en vivo con sus embeddings precomputados:


In [6]:
from aurum_discovery import load_filtered_queries

filtered_queries = load_filtered_queries()
filtered_matrix = embedding_set.matrix("consultas_filtradas")
filtered_ids = embedding_set.identifiers["consultas_filtradas"]
verification_rows = []
for _, query in filtered_queries.iterrows():
    vector = filtered_matrix[filtered_ids.index(query["workload_id"])]
    hits = store.search(vector, top_k=10, brand=query["filter_value"])
    brands = {hit.brand for hit in hits}
    assert brands == {query["filter_value"]}, (query["workload_id"], brands)
    verification_rows.append(
        {"workload_id": query["workload_id"],
         "consulta": query["query_text"][:42],
         "marca": query["filter_value"],
         "resultados": len(hits),
         "todas_cumplen": brands == {query["filter_value"]}}
    )
pd.DataFrame(verification_rows)


,workload_id,consulta,marca,resultados,todas_cumplen
0,FILTER-001,herramienta inalámbrica para perforar,Einhell,10,True
1,FILTER-002,tableta ligera para estudiar y tomar apunt,Apple,10,True
2,FILTER-003,zapatillas cómodas para salir a correr,NIKE,10,True
3,FILTER-004,monitor para trabajar con varias ventanas,SAMSUNG,10,True


### El filtro cambia el universo, no recorta la lista

La comparación siguiente lo demuestra: para la primera consulta filtrada,
el top-10 **global** solo contiene algunos productos de la marca; el top-10
**filtrado** devuelve diez productos de la marca, incluidos varios que el
post-filtrado habría perdido.


In [7]:
first_query = filtered_queries.iloc[0]
vector = filtered_matrix[filtered_ids.index(first_query["workload_id"])]
global_hits = store.search(vector, top_k=10)
filtered_hits = store.search(vector, top_k=10, brand=first_query["filter_value"])
surviving = [hit for hit in global_hits if hit.brand == first_query["filter_value"]]
recovered = {hit.product_id for hit in filtered_hits} - {hit.product_id for hit in surviving}
print(f"Consulta: {first_query['query_text']!r} · marca {first_query['filter_value']}")
print(f"Top-10 global: {len(surviving)}/10 cumplen la marca")
print(f"Top-10 filtrado: 10/10 cumplen; {len(recovered)} productos que el "
      "post-filtrado habría perdido")


Consulta: 'herramienta inalámbrica para perforar' · marca Einhell
Top-10 global: 3/10 cumplen la marca
Top-10 filtrado: 10/10 cumplen; 7 productos que el post-filtrado habría perdido


## Casos límite, uno a uno

**Filtro sin resultados** — lista vacía sin excepción (es una respuesta
válida, no un error):


In [8]:
no_results = store.search(filtered_matrix[0], top_k=10, brand="MarcaInexistenteAurum")
print(f"Marca inexistente -> {no_results!r}")


Marca inexistente -> []


**Colección vacía** — error accionable, no una lista vacía engañosa. Se
demuestra con una colección efímera del propio namespace de la actividad,
que se elimina al terminar con el token de confirmación exacto:


In [9]:
from aurum_discovery import EmptyCollectionError

empty_store = CatalogVectorStore(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
    collection_name="aurum-market-eval-demo-vacia",
    vector_size=embedding_set.configuration.dimension,
    hnsw=run_config.hnsw,
    ef_search=run_config.ef_search,
)
empty_store.ensure_collection()
try:
    empty_store.search(filtered_matrix[0], top_k=5)
except EmptyCollectionError as error:
    print(f"EmptyCollectionError: {error}")
finally:
    empty_store.delete_collection(
        confirmation="DELETE:aurum-market-eval-demo-vacia"
    )
print("Colección efímera eliminada.")


EmptyCollectionError: La colección 'aurum-market-eval-demo-vacia' está vacía. Ingiere el catálogo con `make ingest` antes de buscar.
Colección efímera eliminada.


**Proveedor no disponible** — toda operación (búsqueda, ingesta, lecturas,
borrados) traduce el fallo de transporte en `VectorStoreUnavailableError`
con instrucciones, en lugar de propagar una excepción críptica del SDK.
Se ejercitan las dos rutas que importan en este capítulo — el chequeo de
conexión y la propia búsqueda:


In [10]:
from aurum_discovery import VectorStoreUnavailableError

unreachable_store = CatalogVectorStore(
    url="http://localhost:9999",
    collection_name="aurum-market-eval-inexistente",
    vector_size=embedding_set.configuration.dimension,
    hnsw=run_config.hnsw,
    ef_search=run_config.ef_search,
    timeout_seconds=2.0,
)
for operation_name, operation in (
    ("ping", lambda: unreachable_store.ping()),
    ("search", lambda: unreachable_store.search(filtered_matrix[0], top_k=5)),
):
    try:
        operation()
    except VectorStoreUnavailableError as error:
        print(f"{operation_name} -> VectorStoreUnavailableError: {str(error)[:90]}...")


ping -> VectorStoreUnavailableError: No se puede conectar con Qdrant en http://localhost:9999. Arranca el servicio con `make up...


/home/manu/dev/github.com/manupm87/pontia-bd-vect/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


search -> VectorStoreUnavailableError: La búsqueda contra 'aurum-market-eval-inexistente' falló. Comprueba que Qdrant sigue dispo...


→ Continúa en `actividad_04_operaciones_y_duplicados.ipynb`.
